# Projet Simulation Exponential Backoff

Neil Bonnard 22505907

In [6]:
import random as rd
import matplotlib.pyplot as plt
import numpy as np
from heapq import heappush, heappop

def exp_lambda(λ):
    return np.random.exponential(1 / λ)   #Fonction pour generer temps random selon la loi exponentielle de paramètre λ

def backoff(i, tau):
    return np.random.exponential((2**i) * tau)    #Fonction pour generer temps random selon la loi exponentielle de paramètre (2^i)*ta

#for i in range(10):
#    print(backoff(i, 1))

In [ ]:
def simulate(N, lam, K, tau, T_max):

    echeancier = []
    t = 0
    stations = []
    reussis = []
    canal_libre = True
    log = []
    n_collisions = 0
    n_arrivals = 0

    for i in range(N):      #Initialisation des stations a l'état 1 et pas de paquets dans la file d'attente
        stations.append({"queue_len": 0,
            "state" : 1,
            "attempt_scheduled" : False,
            "is_attempting" : False,
            "end_valid": False})
    
    for i in range(N):      #Initialisation de l'echeancier avec les evenements d'arrivee des paquets a chaque station
        t_arrival = exp_lambda(lam)
        heappush(echeancier, (t_arrival, "ARRIVAL", i))


    while t < T_max: 

        evt = heappop(echeancier)     #Recuperation de l'evenement le plus proche
        t = evt[0]
        station = evt[2]
        match evt[1]:
            case "ARRIVAL":   #Si c'est un evenement d'arrivee de paquet
                t_arrival = t + exp_lambda(lam)
                heappush(echeancier, (t_arrival, "ARRIVAL", station))

                if stations[station]["queue_len"] < K:
                    stations[station]["queue_len"] += 1
                    n_arrivals += 1
                if stations[station]["queue_len"] == 1 and not stations[station]["attempt_scheduled"] and not stations[station]["is_attempting"]:
                    heappush(echeancier, (t, "ATTEMPT", station))
                    stations[station]["attempt_scheduled"] = True
                

            case "ATTEMPT":   #Si c'est un la station essaye d'envoyer un paquet

                if canal_libre:
                    heappush(echeancier, (t+1, "END_TX", station))
                    canal_libre = False
                    stations[station]["is_attempting"] = True
                    stations[station]["end_valid"] = True
                    heappush(log, (t, station, "ATTEMPT"))

                else: 
                    stations[station]["is_attempting"] = False
                    stations[station]["attempt_scheduled"] = True
                    stations[station]["state"] += 1
                    stations[station]["end_valid"] = False
                    t_backoff = t + backoff(stations[station]["state"], tau)
                    heappush(echeancier, (t_backoff, "ATTEMPT", station))
                    for i in range(N):
                        if i != station and stations[i]["is_attempting"]:
                            r = i
                            stations[i]["is_attempting"] = False
                            stations[i]["attempt_scheduled"] = True
                            stations[i]["state"] += 1
                            t_backoff_i = t + backoff(stations[i]["state"], tau)
                            heappush(echeancier, (t_backoff_i, "ATTEMPT", i))
                            stations[i]["end_valid"] = False

                    heappush(log, (t, station, "COLLISION", r))   #On ajoute l'evenement de collision au log
                    n_collisions += 1


            case "END_TX":    #Si un envoi de paquet est terminer
                
                if stations[station]["end_valid"]:   
                    stations[station]["state"] = 1  #On remet la station a l'etat de base
                    stations[station]["queue_len"] -= 1
                    reussis.append((station, t))   #On ajoute le paquet a la liste des paquets reussis avec le temps d'arrivee du paquet
                    stations[station]["is_attempting"] = False
                    stations[station]["attempt_scheduled"] = False
                    if stations[station]["queue_len"] > 0:   #Si il y a encore des paquets dans la file d'attente de la station, on programme un nouvel attempt
                        heappush(echeancier, (t, "ATTEMPT", station))
                        stations[station]["attempt_scheduled"] = True
                    heappush(log, (t, station, "END_TX"))
                n_attempting = sum(1 for s in stations if s["is_attempting"])
                if n_attempting == 0:   #Si aucune station n'est en train de essayer d'envoyer un paquet, on libere le canal
                    canal_libre = True
                else:
                    canal_libre = False
                for station in stations:
                    if station["is_attempting"]:
                        station["is_attempting"] = False
                        station["attempt_scheduled"] = True

    return log, reussis, n_collisions, n_arrivals

In [1]:
import csv

# Paramètres fixes
K = 10
tau = 1
T_max = 10000
lam = 1

# Nombre de stations à tester
Ns = [5, 10, 20, 50, 100]

results = []

for N in Ns:

    print(f"Simulation N = {N}")

    log, successes, n_collisions = simulate(
        N,
        lam,
        K,
        tau,
        T_max
    )

    throughput = successes / T_max

    results.append({
        "N": N,
        "lambda": lam,
        "K": K,
        "tau": tau,
        "T_max": T_max,
        "successes": successes,
        "collisions": n_collisions,
        "throughput": throughput
    })

    print(
        f"N={N} | throughput={throughput:.4f} | collisions={n_collisions}"
    )

# Sauvegarde CSV
with open("results_N.csv", "w", newline="") as csvfile:

    fieldnames = [
        "N",
        "lambda",
        "K",
        "tau",
        "T_max",
        "successes",
        "collisions",
        "throughput"
    ]

    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()

    for row in results:
        writer.writerow(row)

print("Résultats sauvegardés dans results_N.csv")
print(log)

Simulation N = 5


NameError: name 'simulate' is not defined

In [10]:
import csv

# Paramètres fixes
N = 20
K = 10
tau = 1
T_max = 10000

# Lambdas à tester
lambdas = [0.01, 0.05, 0.1, 0.2, 0.5, 1, 2, 5]

results = []

for lam in lambdas:

    print(f"Simulation lambda = {lam}")

    log, successes, n_collisions, n_arrivals= simulate(
        N,
        lam,
        K,
        tau,
        T_max
    )

    throughput = successes / T_max

    results.append({
        "lambda": lam,
        "N": N,
        "K": K,
        "tau": tau,
        "T_max": T_max,
        "successes": successes,
        "collisions": n_collisions,
        "throughput": throughput,
        "arrivals": n_arrivals
    })

    print(
        f"lambda={lam} | throughput={throughput:.4f} | collisions={n_collisions}"
    )

# Sauvegarde CSV
with open("results_lambda.csv", "w", newline="") as csvfile:

    fieldnames = [
        "lambda",
        "N",
        "K",
        "tau",
        "T_max",
        "successes",
        "collisions",
        "throughput",
        "arrivals"
    ]

    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()

    for row in results:
        writer.writerow(row)

print("Résultats sauvegardés dans results_lambda.csv")
print(log)

Simulation lambda = 0.01
lambda=0.01 | throughput=0.2037 | collisions=1156
Simulation lambda = 0.05
lambda=0.05 | throughput=0.2823 | collisions=1654
Simulation lambda = 0.1
lambda=0.1 | throughput=0.3361 | collisions=1616
Simulation lambda = 0.2
lambda=0.2 | throughput=0.3889 | collisions=1466
Simulation lambda = 0.5
lambda=0.5 | throughput=0.5136 | collisions=819
Simulation lambda = 1
lambda=1 | throughput=0.8121 | collisions=578
Simulation lambda = 2
lambda=2 | throughput=0.8584 | collisions=495
Simulation lambda = 5
lambda=5 | throughput=0.8732 | collisions=430
Résultats sauvegardés dans results_lambda.csv
[(0.005989743077134373, 9, 'ATTEMPT'), (0.008974561923050633, 10, 'COLLISION', 9), (0.015148847743331424, 4, 'COLLISION', 9), (0.019663298600968538, 1, 'COLLISION', 9), (0.023105209282175923, 5, 'COLLISION', 9), (0.028509975110201143, 15, 'COLLISION', 9), (0.03269975428058358, 3, 'COLLISION', 9), (0.05532046203329882, 18, 'COLLISION', 9), (0.060868788185934, 12, 'COLLISION', 9), 